# 데이터 다운로드 (S3 → SageMaker 노트북 로컬)

학습 전에 **한 번만** 실행한다. S3에서 이미지를 노트북 로컬 디스크로 복사해
학습 때 S3 네트워크 왕복을 없앤다 (학습 속도 대폭 향상).

- 원본: `s3://aws-say3-dataset1/pneumonia-xray/chest_xray/` (약 1.2GB, 5,856장)
- 저장: 노트북 로컬 `./data/chest_xray/`
- 인스턴스를 Stop해도 로컬(EBS)에 남지만, Space를 지우면 사라짐 → 그때 다시 받으면 됨
- 이 다운로드엔 GPU 불필요 (CPU 인스턴스에서 받아도 됨)

## 다운로드 (aws s3 sync, 이미 있으면 바뀐 것만 받음)

In [ ]:
import os, subprocess, time

LOCAL_DATA = os.path.abspath('./data/chest_xray')
os.makedirs(LOCAL_DATA, exist_ok=True)
S3_SRC = 's3://aws-say3-dataset1/pneumonia-xray/chest_xray/'

print('다운로드 시작:', S3_SRC, '->', LOCAL_DATA)
t0 = time.time()
# sync: 이미 받은 파일은 건너뛰므로 재실행해도 안전
subprocess.run(['aws', 's3', 'sync', S3_SRC, LOCAL_DATA], check=True)
print(f'완료. 소요 {time.time()-t0:.0f}s')

## 확인 — 파일 개수/구조

In [ ]:
import glob
n = len(glob.glob(os.path.join(LOCAL_DATA, '**', '*.jpeg'), recursive=True))
print('받은 jpeg 수:', n, '(기대: 5856)')
for sp in ['train', 'val', 'test']:
    for cl in ['NORMAL', 'PNEUMONIA']:
        p = os.path.join(LOCAL_DATA, sp, cl)
        c = len(glob.glob(os.path.join(p, '*.jpeg'))) if os.path.isdir(p) else 0
        print(f'  {sp}/{cl}: {c}')
print('\nLOCAL_DATA 경로(학습 노트북에서 이 경로 사용):')
print(' ', LOCAL_DATA)